## Tutorial For SGLang Basic Usage

下面只放直接可读的 CLI 命令。服务启动命令会常驻运行，建议在终端里执行。

### 1. 当前 SGLang / Torch 环境版本

```bash
python3 --version
python3 -m pip show sglang sglang-router torch transformers
python3 - <<'PY'
import importlib.metadata as md

for name in ["sglang", "sglang-router", "torch", "transformers"]:
    try:
        print(f"{name}: {md.version(name)}")
    except md.PackageNotFoundError:
        print(f"{name}: not installed")

try:
    import torch
    print("torch.version.cuda:", torch.version.cuda)
    print("cuda_available:", torch.cuda.is_available())
    print("cuda_device_count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"gpu_{i}: {props.name}, {props.total_memory / 1024**3:.1f} GiB")
except Exception as exc:
    print("torch import check failed:", repr(exc))
PY
nvidia-smi
```

### 2. 当前 SGLang CLI 参数有哪些

完整参数以当前安装版本为准：

```bash
python3 -m sglang.launch_server --help
sglang --help
sglang serve --help
```

重点关注：

| 类别 | 重点参数 |
| --- | --- |
| 模型与 tokenizer | `--model-path` / `--model`, `--tokenizer-path`, `--trust-remote-code`, `--context-length` |
| 服务入口 | `--host`, `--port`, `--served-model-name`, `--api-key` |
| dtype / 量化 | `--dtype`, `--quantization`, `--kv-cache-dtype` |
| 内存与调度 | `--mem-fraction-static`, `--max-total-tokens`, `--max-running-requests`, `--max-prefill-tokens`, `--chunked-prefill-size`, `--schedule-policy` |
| 并行 | `--tp` / `--tp-size`, `--pp-size`, `--dp-size`, `--enable-dp-attention`, `--ep` / `--ep-size`, `--attention-context-parallel-size` / `--attn-cp-size` |
| overlap / 性能 | `--disable-overlap-schedule`, `--enable-two-batch-overlap`, `--enable-single-batch-overlap`, `--mamba-scheduler-strategy`, `--page-size`, `--attention-backend` |
| Qwen / 工具调用 | `--reasoning-parser qwen3`, `--tool-call-parser qwen3_coder`, `--sampling-defaults` |
| MoE | `--moe-a2a-backend`, `--moe-runner-backend`, `--enable-eplb`, `--deepep-mode` |

### 3. 单卡运行 Qwen3.5-0.8B：最普通方式

Qwen3.5 原生上下文很长，普通单卡容易因为 KV cache OOM；这里先用 `--context-length 32768` 做 smoke test。

```bash
CUDA_VISIBLE_DEVICES=0 python3 -m sglang.launch_server --model-path Qwen/Qwen3.5-0.8B --host 0.0.0.0 --port 30000 --mem-fraction-static 0.8 --context-length 32768
```

连通性测试：

```bash
curl -sS http://127.0.0.1:30000/v1/chat/completions -H 'Content-Type: application/json' -d '{"model":"Qwen/Qwen3.5-0.8B","messages":[{"role":"user","content":"用一句话解释 SGLang 是什么。"}],"max_tokens":128,"temperature":0.2}'
```

### 4. 单卡运行 Qwen3.5-0.8B：使用 overlap

Qwen3.5 的 hybrid Gated Delta Networks / Mamba cache 路径可用 `--mamba-scheduler-strategy extra_buffer --page-size 64` 打开 overlap scheduler。该模式更适合 NVIDIA + FLA kernel；如果遇到 kernel 或显存问题，退回上一节普通命令。

```bash
CUDA_VISIBLE_DEVICES=0 python3 -m sglang.launch_server --model-path Qwen/Qwen3.5-0.8B --host 0.0.0.0 --port 30000 --mem-fraction-static 0.8 --context-length 32768 --mamba-scheduler-strategy extra_buffer --page-size 64
```

### 5. 双卡并行 / 5D 并行 smoke test

双卡无法让 TP、PP、DP/DPA、EP、CP 五个维度全部大于 1。下面是双卡 smoke test：让 `tp=2`、`dp-size=2`、`ep=2` 覆盖 TP + DPA + EP；`pp-size=1`、`attention-context-parallel-size=1` 只保留参数位。

模型建议：为了真实覆盖 EP/MoE，默认用 `Qwen/Qwen3.5-35B-A3B-FP8`。建议 2x80GB；2x48GB 可先把 `--context-length` 降到 `8192` 或 `16384`。如果只想测试双卡 TP，不关心 EP/MoE，可换成更小的 dense 模型，但那不算真实 EP 测试。

```bash
CUDA_VISIBLE_DEVICES=0,1 python3 -m sglang.launch_server --model-path Qwen/Qwen3.5-35B-A3B-FP8 --host 0.0.0.0 --port 30000 --mem-fraction-static 0.8 --context-length 32768 --tp 2 --dp-size 2 --enable-dp-attention --ep 2 --moe-a2a-backend none --moe-runner-backend auto --enable-two-batch-overlap --mamba-scheduler-strategy extra_buffer --page-size 64 --pp-size 1 --attention-context-parallel-size 1 --reasoning-parser qwen3
```

真正让五个并行维度都大于 1，需要更多 GPU/rank。例如 `tp=2`、`pp=2`、`dp-size=2`、`ep=2`、`attention-context-parallel-size=2` 这类模板已经不是双卡范畴，应在 8 卡或更多资源上验证，并优先选择 MoE 模型。